# Load the synthetic Purview export

This notebook loads the CSV files from the repo's `data/purview-export/` folder into Delta tables in the `purview_export` lakehouse, one table per file, then checks that every table arrived.

Before you run it:

1. Pin `purview_export` as the notebook's default lakehouse, then restart the session if one is running.
2. In the lakehouse explorer, create the folder `purview-export` under `Files` and upload the CSV files into it.

The data describes a fictional organization. `custom_access_use_case` has a header and no rows, so its table is empty by design. Running the notebook again replaces the tables.

In [ ]:
source_folder = "Files/purview-export"

In [ ]:
expected_tables = [
    "access_policy_provisioning_state",
    "access_policy_resource_type",
    "access_policy_set",
    "business_domain",
    "critical_data_element",
    "custom_access_use_case",
    "data_asset",
    "data_asset_domain_assignment",
    "data_asset_owner",
    "data_asset_owner_assignment",
    "data_product",
    "data_product_asset_assignment",
    "data_product_business_domain_assignment",
    "data_product_owner",
    "data_product_status",
    "data_product_type",
    "data_product_update_frequency",
    "data_quality_rule",
    "data_quality_rule_type",
    "data_subscription_request",
    "glossary_term",
    "glossary_term_business_domain_assignment",
    "health_action",
    "key_result",
    "objective",
    "policy_set_approver",
    "relationship",
]
empty_by_design = {"custom_access_use_case"}

In [ ]:
from pyspark.sql.utils import AnalysisException

row_counts = {}
missing = []

for table in expected_tables:
    path = f"{source_folder}/{table}.csv"
    try:
        df = (
            spark.read.option("header", True)
            .option("inferSchema", True)
            .option("escape", '"')
            .csv(path)
        )
    except AnalysisException:
        missing.append(table)
        continue
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    row_counts[table] = spark.table(table).count()
    print(f"{table}: {row_counts[table]} rows")

In [ ]:
empty = sorted(t for t, n in row_counts.items() if n == 0 and t not in empty_by_design)

print(f"Loaded {len(row_counts)} of {len(expected_tables)} tables.")
if missing or empty:
    raise ValueError(
        f"Missing files: {', '.join(missing) or 'none'}. "
        f"Unexpectedly empty tables: {', '.join(empty) or 'none'}."
    )
print("Every table loaded.")